# 1. Init — 环境与建库

准备 `scope40_easy`：

1. 下载 Foldseek（Linux AVX2）
2. 获取 SCOPe40 runtime（优先本地 `../scope40_hf_dataset/output/scope40_runtime`；否则从 GitHub 下载 tarball）
3. 链接各模型对 `DB_aa.fasta` 预测得到的 `*aa2di.fasta`（默认 `../new_scope40/fasta`）
4. 构建各方法 Foldseek DB → `work/dbs/`

GitHub 备用包：
`https://github.com/caijihuize/scope40_hf_dataset/raw/master/output/scope40_runtime.tar.gz`

前置：
```bash
conda activate ESM3_3Di_5090
cd /hpcfs/fhome/caihuize/scope40_easy
jupyter lab
```


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'lib').is_dir():
    raise SystemExit(f'请在项目根目录启动 notebook，当前: {ROOT}')
sys.path.insert(0, str(ROOT))

from lib import config
from lib.setup_env import (
    download_foldseek,
    link_scope40_runtime,
    link_prediction_fastas,
    check_environment,
)
from lib.build_db import build_all

SKIP_EXISTING = True
FORCE_REDOWNLOAD_FOLDSEEK = False
FORCE_RELINK_RUNTIME = False

print('local runtime src:', config.HF_RUNTIME_SRC)
print('GitHub backup URL:', config.GITHUB_RUNTIME_URL)
print('pred fasta src:', config.PRED_FASTA_SRC)


## 下载 Foldseek（Linux AVX2）

从 GitHub release `10-941cd33` 下载并解压为 `./foldseek/bin/foldseek`。


In [ ]:
download_foldseek(skip_existing=SKIP_EXISTING, force=FORCE_REDOWNLOAD_FOLDSEEK)
print('bin:', config.FOLDSEEK_BIN, 'exists=', config.FOLDSEEK_BIN.is_file())


## 链接 / 下载 SCOPe40 运行包 + 模型预测 3Di

- **Runtime**：优先软链本地 `../scope40_hf_dataset/output/scope40_runtime`；
  若本地不存在，则 wget GitHub 备用包并解压到 `data/scope40_runtime/`：
  `https://github.com/caijihuize/scope40_hf_dataset/raw/master/output/scope40_runtime.tar.gz`
- **模型预测 di.fasta**：软链到 `data/pred_fasta/`（默认来源 `../new_scope40/fasta`，可用 `PRED_FASTA_DIR` 覆盖）


In [ ]:
link_scope40_runtime(force=FORCE_RELINK_RUNTIME)
link_prediction_fastas()

status = check_environment()
missing = [k for k, ok in status.items() if not ok]
if missing:
    raise SystemExit(f'环境未就绪，缺失: {missing}')
print('\n环境检查通过。')


## 构建实验数据库

- `foldseek`：链接运行包中的结构真值 FoldseekDB
- 其余方法：`DB_aa.fasta` + 预测 `*aa2di.fasta` → `tsv2db`

产物：`work/dbs/`


In [ ]:
dbs = build_all(skip_existing=SKIP_EXISTING)
for key, path in dbs.items():
    print(f'{key:12s} → {path}  exists={path.is_file()}')
print('\nInit 完成。下一步打开 2.benchmark.ipynb')
